In [1]:
# NORTHSTAR — Solace Browser: Demo Page QA
# DNA: demo_qa = interactive(try_buttons+modals) x features(OAuth3+Recipes+BYOK+Evidence) x honesty(no_fabrication) x a11y(WCAG)
# Page: demo.html
# Towers: T1(Masters) T2(Customers) T6(Hackers) T7(Kids) T8(Elders) T23(Web)
# Auth: 65537 | File-based probes — reads HTML directly (no running server)
# REAL assertions. No mocks. No stubs.

import json, re, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "solace-browser-demo-page-qa"
NOTEBOOK_PATH = "notebooks/qa/29-demo-page-qa.ipynb"
PROJECT = "solace-browser"
MIN_SCORE = 70

BROWSER = Path("/home/phuc/projects/solace-browser")
WEB = BROWSER / "web"
DEMO_HTML = WEB / "demo.html"

results = []

def record(probe_id, name, passed, detail="", tower_ref=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status,
                    "detail": detail, "tower_ref": tower_ref})
    print(f"  {status}: {name}" + (f" -- {detail}" if detail and not passed else ""))

assert DEMO_HTML.exists(), f"MISSING: {DEMO_HTML}"
demo_html = DEMO_HTML.read_text(encoding="utf-8")

print(f"BOOTSTRAP: demo.html loaded ({len(demo_html)} chars)")
print(f"File: {DEMO_HTML}")

BOOTSTRAP: demo.html loaded (31036 chars)
File: /home/phuc/projects/solace-browser/web/demo.html


In [2]:
# P1: Demo has interactive elements (try-it buttons, preview modals)
print("=== P1: Interactive Elements ===")

# Try-it / try-btn buttons
try_buttons = re.findall(r'class="[^"]*try-btn[^"]*"', demo_html)
record("P1-try-buttons", f"demo.html has try-it buttons ({len(try_buttons)} found)",
       len(try_buttons) >= 1,
       detail="No try-btn elements found" if not try_buttons else "",
       tower_ref="T1,T2,T23")

# Preview modals
has_modal = bool(re.search(r'preview-modal|preview-overlay|modal', demo_html))
record("P1-preview-modal", "demo.html has preview modal/overlay", has_modal,
       detail="No preview-modal or preview-overlay found" if not has_modal else "",
       tower_ref="T1,T2,T23")

# Buttons are actual <button> elements (not just <a> styled as buttons)
button_tags = re.findall(r'<button[^>]*class="[^"]*try-btn', demo_html)
record("P1-semantic-buttons", f"try-it elements use <button> tags ({len(button_tags)} found)",
       len(button_tags) >= 1,
       detail="try-btn elements should use <button> for accessibility" if not button_tags else "",
       tower_ref="T1,T8")

# Interactive JS (onclick, addEventListener, or script references)
has_js_interaction = bool(re.search(r'addEventListener|onclick|<script', demo_html))
record("P1-js-interaction", "demo.html has JavaScript for interactivity", has_js_interaction,
       detail="No JS interaction found" if not has_js_interaction else "",
       tower_ref="T1,T23")

passed = sum(1 for r in results if r["id"].startswith("P1") and r["status"] == "PASS")
total_p1 = sum(1 for r in results if r["id"].startswith("P1"))
print(f"\nP1 complete: {passed}/{total_p1} passed")

=== P1: Interactive Elements ===
  PASS: demo.html has try-it buttons (4 found)
  PASS: demo.html has preview modal/overlay
  PASS: try-it elements use <button> tags (2 found)
  PASS: demo.html has JavaScript for interactivity

P1 complete: 4/4 passed


In [3]:
# P2: Demo mentions key features (OAuth3, Recipes, BYOK, Evidence)
print("=== P2: Key Feature Mentions ===")

REQUIRED_FEATURES = {
    "OAuth3": r'[Oo][Aa]uth3|oauth3',
    "Recipes": r'[Rr]ecipe',
    "BYOK": r'BYOK|[Bb]ring\s*[Yy]our\s*[Oo]wn\s*[Kk]ey|byok',
    "Evidence": r'[Ee]vidence',
}

for feature_name, pattern in REQUIRED_FEATURES.items():
    found = bool(re.search(pattern, demo_html))
    record(f"P2-{feature_name.lower()}", f"demo.html mentions {feature_name}", found,
           detail=f"No mention of {feature_name} found" if not found else "",
           tower_ref="T1,T2")

# Preview mention (core concept)
has_preview = bool(re.search(r'[Pp]review', demo_html))
record("P2-preview", "demo.html mentions Preview concept", has_preview,
       detail="No mention of Preview found" if not has_preview else "",
       tower_ref="T1,T2")

passed = sum(1 for r in results if r["id"].startswith("P2") and r["status"] == "PASS")
total_p2 = sum(1 for r in results if r["id"].startswith("P2"))
print(f"\nP2 complete: {passed}/{total_p2} passed")

=== P2: Key Feature Mentions ===
  PASS: demo.html mentions OAuth3
  PASS: demo.html mentions Recipes
  PASS: demo.html mentions BYOK
  PASS: demo.html mentions Evidence
  PASS: demo.html mentions Preview concept

P2 complete: 5/5 passed


In [4]:
# P3: Demo has QA checklist (accessibility / usability / feedback)
print("=== P3: QA Checklist ===")

# Accessibility: has alt text on images, aria labels, semantic headings
img_tags = re.findall(r'<img\s[^>]*>', demo_html)
imgs_with_alt = [t for t in img_tags if re.search(r'alt="[^"]*"', t)]
record("P3-img-alt", f"demo.html images have alt text ({len(imgs_with_alt)}/{len(img_tags)})",
       len(img_tags) == 0 or len(imgs_with_alt) == len(img_tags),
       detail=f"{len(img_tags) - len(imgs_with_alt)} images missing alt text" if imgs_with_alt != img_tags and img_tags else "",
       tower_ref="T8")

# Headings hierarchy (h1 followed by h2s, no skipping)
headings = re.findall(r'<h([1-6])[^>]*>', demo_html)
has_h1 = '1' in headings
record("P3-heading-h1", "demo.html has an <h1> heading", has_h1,
       detail="No <h1> tag found" if not has_h1 else "",
       tower_ref="T8,T23")

# Feedback form or link
has_feedback = bool(re.search(r'feedback|contact|support|form|mailto:', demo_html, re.IGNORECASE))
record("P3-feedback", "demo.html has feedback/contact mechanism", has_feedback,
       detail="No feedback form, contact link, or mailto found" if not has_feedback else "",
       tower_ref="T1,T2")

# CSP meta tag
has_csp = bool(re.search(r'Content-Security-Policy', demo_html))
record("P3-csp", "demo.html has Content-Security-Policy meta tag", has_csp,
       detail="Missing CSP meta tag" if not has_csp else "",
       tower_ref="T6")

# viewport meta for mobile
has_viewport = bool(re.search(r'<meta\s+name="viewport"', demo_html))
record("P3-viewport", "demo.html has viewport meta tag", has_viewport,
       detail="Missing viewport meta tag" if not has_viewport else "",
       tower_ref="T23")

passed = sum(1 for r in results if r["id"].startswith("P3") and r["status"] == "PASS")
total_p3 = sum(1 for r in results if r["id"].startswith("P3"))
print(f"\nP3 complete: {passed}/{total_p3} passed")

=== P3: QA Checklist ===
  PASS: demo.html images have alt text (0/0)
  PASS: demo.html has an <h1> heading
  PASS: demo.html has feedback/contact mechanism
  PASS: demo.html has Content-Security-Policy meta tag
  PASS: demo.html has viewport meta tag

P3 complete: 5/5 passed


In [5]:
# P4: No fabricated data or fake metrics in demo
print("=== P4: No Fabricated Data ===")

# No fabricated testimonials
testimonial_patterns = [
    r'"[^"]{20,}"\s*[\u2014\u2013\-]\s*[A-Z][a-z]+ [A-Z]',  # "Quote" -- Firstname L
    r'customer\s+said|user\s+said|testimonial',
]
fabricated_found = []
for pat in testimonial_patterns:
    matches = re.findall(pat, demo_html)
    fabricated_found.extend(matches)

record("P4-no-testimonials", "demo.html has no fabricated testimonials",
       len(fabricated_found) == 0,
       detail=f"Found {len(fabricated_found)} potential fabricated testimonials" if fabricated_found else "",
       tower_ref="T1,T7")

# No inflated user count metrics (e.g., "10k users", "50,000 customers")
inflated_metrics = re.findall(r'\d{1,3}[,.]?\d{3,}\+?\s*(users|customers|downloads|companies)', demo_html, re.IGNORECASE)
record("P4-no-inflated-metrics", "demo.html has no inflated user/download counts",
       len(inflated_metrics) == 0,
       detail=f"Found: {inflated_metrics}" if inflated_metrics else "",
       tower_ref="T1,T7")

# No fake speed claims ("10x faster", "99.9% uptime")
fake_claims = re.findall(r'\d+x\s+faster|\d+\.\d+%\s+uptime|guaranteed\s+\d+%', demo_html, re.IGNORECASE)
record("P4-no-fake-claims", "demo.html has no unsubstantiated speed/uptime claims",
       len(fake_claims) == 0,
       detail=f"Found: {fake_claims}" if fake_claims else "",
       tower_ref="T1,T7")

# No fake screenshots or stock photos (check for suspicious image names)
stock_imgs = re.findall(r'(stock|fake|placeholder|lorem|dummy)\.(png|jpg|svg)', demo_html, re.IGNORECASE)
record("P4-no-stock-images", "demo.html has no stock/placeholder images",
       len(stock_imgs) == 0,
       detail=f"Found: {stock_imgs}" if stock_imgs else "",
       tower_ref="T1,T7")

passed = sum(1 for r in results if r["id"].startswith("P4") and r["status"] == "PASS")
total_p4 = sum(1 for r in results if r["id"].startswith("P4"))
print(f"\nP4 complete: {passed}/{total_p4} passed")

=== P4: No Fabricated Data ===
  PASS: demo.html has no fabricated testimonials
  PASS: demo.html has no inflated user/download counts
  PASS: demo.html has no unsubstantiated speed/uptime claims
  PASS: demo.html has no stock/placeholder images

P4 complete: 4/4 passed


In [6]:
# P5: Evidence summary — aggregate results, compute score, seal hash
print("=== P5: Evidence Summary ===")

total = len(results)
passed = sum(1 for r in results if r["status"] == "PASS")
failed = sum(1 for r in results if r["status"] == "FAIL")
score = round(100 * passed / total, 1) if total > 0 else 0.0

print(f"\n{'='*60}")
print(f"  NOTEBOOK: {NOTEBOOK_PATH}")
print(f"  PROJECT:  {PROJECT}")
print(f"  PAGE:     demo.html")
print(f"  TOTAL:    {total} probes")
print(f"  PASSED:   {passed}")
print(f"  FAILED:   {failed}")
print(f"  SCORE:    {score}%  (min required: {MIN_SCORE}%)")
print(f"{'='*60}")

if failed > 0:
    print("\nFAILURES:")
    for r in results:
        if r["status"] == "FAIL":
            print(f"  FAIL: {r['id']} \u2014 {r['name']}")
            if r["detail"]:
                print(f"        {r['detail']}")

# Seal
evidence = {
    "notebook": NOTEBOOK_PATH,
    "northstar": NORTHSTAR,
    "project": PROJECT,
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "total": total,
    "passed": passed,
    "failed": failed,
    "score_pct": score,
    "min_score": MIN_SCORE,
    "verdict": "PASS" if score >= MIN_SCORE else "FAIL",
    "results": results,
}

seal = hashlib.sha256(json.dumps(evidence, sort_keys=True).encode()).hexdigest()[:16]
evidence["seal"] = seal

print(f"\n  VERDICT: {evidence['verdict']}")
print(f"  SEAL:    {seal}")
print(f"  AUTH:    65537")

=== P5: Evidence Summary ===

  NOTEBOOK: notebooks/qa/29-demo-page-qa.ipynb
  PROJECT:  solace-browser
  PAGE:     demo.html
  TOTAL:    18 probes
  PASSED:   18
  FAILED:   0
  SCORE:    100.0%  (min required: 70%)

  VERDICT: PASS
  SEAL:    a56d25727168a46c
  AUTH:    65537
